# Voice Cleanliness Model - QualiSpeech

This notebook starts the speech-cleanliness phase of the graduation project. It uses the QualiSpeech dataset as the supervision source for detecting whether an audio recording is clean enough to pass into ASR and then into the narrative coherence checker.

QualiSpeech contains multidimensional speech-quality labels: speed, naturalness, background noise, distortion, listening effort, continuity, and overall quality. It also includes natural-language descriptions of noise, distortion, pauses, voice feeling, and overall quality reasoning. This matches the thesis literature review, which argues that the audio module should be non-intrusive and multidimensional, inspired by NISQA, MOSNet, UTMOS, and QualiSpeech.

## Setup

If needed, install the missing packages in your environment before running the notebook:

```powershell
pip install datasets soundfile librosa scikit-learn matplotlib transformers accelerate
```

Important dataset note: the QualiSpeech card states that some BVCC-related audio cannot be redistributed directly due to licensing restrictions. If audio files are missing, follow the dataset card's `download_bvcc.sh` and `merge_data.sh` instructions before full audio training.

In [1]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASET_NAME = "tsinghua-ee/QualiSpeech"
OUTPUT_DIR = Path("outputs/voice_cleanliness_qualispeech")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_TARGETS = [
    "Speed",
    "Naturalness",
    "Background noise",
    "Distortion",
    "Listening effort",
    "Continuity",
    "Overall quality",
]

TEXT_FIELDS = [
    "Feeling of voice",
    "Noise Description",
    "Distortion description",
    "Unnatural pause",
    "Natural language description",
]

## Load Dataset Metadata

The Hugging Face dataset viewer reports approximately 10.6k training rows, 2.17k validation rows, and 1.85k test rows. The first run can start with metadata inspection before audio feature extraction.

In [2]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME)
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'Speed', 'Naturalness', 'Background noise', 'Distortion', 'Listening effort', 'Continuity', 'Overall quality', 'Feeling of voice', 'Noise Description', 'Distortion description', 'Unnatural pause', 'Natural language description'],
        num_rows: 10558
    })
    validation: Dataset({
        features: ['id', 'Speed', 'Naturalness', 'Background noise', 'Distortion', 'Listening effort', 'Continuity', 'Overall quality', 'Feeling of voice', 'Noise Description', 'Distortion description', 'Unnatural pause', 'Natural language description'],
        num_rows: 2167
    })
    test: Dataset({
        features: ['id', 'Speed', 'Naturalness', 'Background noise', 'Distortion', 'Listening effort', 'Continuity', 'Overall quality', 'Feeling of voice', 'Noise Description', 'Distortion description', 'Unnatural pause', 'Natural language description'],
        num_rows: 1852
    })
})

In [3]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

print(train_df.shape, val_df.shape, test_df.shape)
display(train_df.head())
display(train_df[NUMERIC_TARGETS].describe())

(10558, 13) (2167, 13) (1852, 13)


,id,Speed,Naturalness,Background noise,Distortion,Listening effort,Continuity,Overall quality,Feeling of voice,Noise Description,Distortion description,Unnatural pause,Natural language description
0,sys01tts-0000000.wav,2,5,5,5,5,5,5,A woman's voice with a rising and falling tone...,Not noticeable,Not distorted,Very smooth,The speech sample exhibits exceptional quality...
1,sys01tts-0000001.wav,2,5,5,5,5,5,5,A woman's euphonic voice with a plain tone tha...,Not noticeable,Not distorted,Very smooth,The speech sample exhibits an impressive level...
2,sys01tts-0000002.wav,2,5,3,5,5,5,4,A woman's voice with a plain tone that sounds ...,There is background noise similar to whistle n...,Not distorted,Very smooth,The speech sample presents a moderate level of...
3,sys01tts-0000003.wav,3,5,4,5,5,5,4,A middle-aged man's deep voice with a plain to...,There is outdoor background noise in the audio...,Not distorted,Very smooth,The speech sample presents a mostly pleasant l...
4,sys01tts-0000004.wav,3,5,3,5,5,5,4,A woman's voice with a plain tone that sounds ...,There is outdoor background noise similar to b...,Not distorted,Very smooth,The speech sample presents a moderate level of...


,Speed,Naturalness,Background noise,Distortion,Listening effort,Continuity,Overall quality
count,10558.000000,10558.000000,10558.000000,10558.000000,10558.000000,10558.000000,10558.000000
mean,3.101250,2.998579,4.354802,3.683842,3.917598,4.123887,3.210646
std,0.530474,1.207746,1.041204,1.190354,1.176332,1.154453,0.965715
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,3.000000,2.000000,4.000000,3.000000,3.000000,3.000000,3.000000
50%,3.000000,3.000000,5.000000,4.000000,4.000000,5.000000,3.000000
75%,3.000000,4.000000,5.000000,5.000000,5.000000,5.000000,4.000000
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


## Cleanliness Target Definition

The project needs a binary gate for the full pipeline: should this audio be accepted for transcription, or should the user be asked to re-record?

The model will still predict numerical quality scores, but the decision label is derived as:

`clean_for_asr = noise >= 4 and distortion >= 4 and continuity >= 4 and listening_effort >= 4 and overall_quality >= 4`

This is intentionally strict because poor audio can damage the ASR transcript, and a poor transcript can later cause false coherence errors.

In [4]:
def add_cleanliness_label(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["clean_for_asr"] = (
        (df["Background noise"] >= 4)
        & (df["Distortion"] >= 4)
        & (df["Continuity"] >= 4)
        & (df["Listening effort"] >= 4)
        & (df["Overall quality"] >= 4)
    ).astype(int)
    return df

train_df = add_cleanliness_label(train_df)
val_df = add_cleanliness_label(val_df)
test_df = add_cleanliness_label(test_df)

display(train_df["clean_for_asr"].value_counts(normalize=True).rename("train proportion"))
display(val_df["clean_for_asr"].value_counts(normalize=True).rename("validation proportion"))
display(test_df["clean_for_asr"].value_counts(normalize=True).rename("test proportion"))

clean_for_asr
0    0.717181
1    0.282819
Name: train proportion, dtype: float64

clean_for_asr
0    0.755884
1    0.244116
Name: validation proportion, dtype: float64

clean_for_asr
0    0.617711
1    0.382289
Name: test proportion, dtype: float64

## Metadata/Text Baseline

This is not the final audio model. It is a sanity baseline that uses the descriptive text fields to verify that the labels are learnable and that the train/validation/test pipeline works. The real deployed model should use audio features, because user recordings will not arrive with human-written quality descriptions.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import accuracy_score, f1_score, classification_report, mean_absolute_error, r2_score

def combine_text_fields(df: pd.DataFrame) -> pd.Series:
    available = [field for field in TEXT_FIELDS if field in df.columns]
    return df[available].fillna("").agg(" ".join, axis=1)

X_train_text = combine_text_fields(train_df)
X_val_text = combine_text_fields(val_df)
X_test_text = combine_text_fields(test_df)

y_train_clean = train_df["clean_for_asr"]
y_val_clean = val_df["clean_for_asr"]
y_test_clean = test_df["clean_for_asr"]

text_gate = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
])

text_gate.fit(X_train_text, y_train_clean)
val_pred = text_gate.predict(X_val_text)
test_pred = text_gate.predict(X_test_text)

print("Validation F1:", f1_score(y_val_clean, val_pred))
print("Test F1:", f1_score(y_test_clean, test_pred))
print(classification_report(y_test_clean, test_pred, target_names=["Re-record", "Clean for ASR"]))

Validation F1: 0.9574861367837338
Test F1: 0.9538249483115093
               precision    recall  f1-score   support

    Re-record       0.99      0.96      0.97      1144
Clean for ASR       0.93      0.98      0.95       708

     accuracy                           0.96      1852
    macro avg       0.96      0.97      0.96      1852
 weighted avg       0.96      0.96      0.96      1852



## Audio Availability Check

QualiSpeech rows include audio paths/IDs, but some source audio may require separate construction because of licensing. This cell checks whether the dataset object exposes an audio column or local files that can be read.

In [6]:
print(train_df.columns.tolist())

possible_audio_columns = [c for c in train_df.columns if "audio" in c.lower() or "path" in c.lower() or c.lower() in {"id", "file", "filename"}]
possible_audio_columns

['id', 'Speed', 'Naturalness', 'Background noise', 'Distortion', 'Listening effort', 'Continuity', 'Overall quality', 'Feeling of voice', 'Noise Description', 'Distortion description', 'Unnatural pause', 'Natural language description', 'clean_for_asr']


['id']

## Practical Audio Feature Baseline

If the waveform files are available locally, this baseline extracts standard acoustic features with `librosa` and trains classical models. These features are not as strong as NISQA-style CNN/self-attention or wav2vec-style transfer learning, but they are interpretable and useful as a first baseline.

Features include duration, RMS energy, zero-crossing rate, spectral centroid, bandwidth, rolloff, flatness, and MFCC statistics.

In [7]:
AUDIO_ROOT = Path("data/QualiSpeech/wav_part1/wav")
AUDIO_PATH_COLUMN = "id"

def resolve_audio_path(row, split: str) -> Path:
    value = str(row[AUDIO_PATH_COLUMN])
    filename = value if value.endswith(".wav") else f"{value}.wav"
    return AUDIO_ROOT / split / filename

sample_paths = [resolve_audio_path(row, "train") for _, row in train_df.head(5).iterrows()]
for path in sample_paths:
    print(path, "exists=", path.exists())


data\QualiSpeech\wav_part1\wav\train\sys01tts-0000000.wav exists= True
data\QualiSpeech\wav_part1\wav\train\sys01tts-0000001.wav exists= True
data\QualiSpeech\wav_part1\wav\train\sys01tts-0000002.wav exists= True
data\QualiSpeech\wav_part1\wav\train\sys01tts-0000003.wav exists= True
data\QualiSpeech\wav_part1\wav\train\sys01tts-0000004.wav exists= True


In [8]:
import librosa

def extract_audio_features(path: Path, sr: int = 16000) -> dict:
    y, sr = librosa.load(path, sr=sr, mono=True)
    duration = len(y) / sr
    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    flatness = librosa.feature.spectral_flatness(y=y)[0]
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    features = {
        "duration": duration,
        "rms_mean": float(np.mean(rms)),
        "rms_std": float(np.std(rms)),
        "zcr_mean": float(np.mean(zcr)),
        "zcr_std": float(np.std(zcr)),
        "centroid_mean": float(np.mean(centroid)),
        "centroid_std": float(np.std(centroid)),
        "bandwidth_mean": float(np.mean(bandwidth)),
        "bandwidth_std": float(np.std(bandwidth)),
        "rolloff_mean": float(np.mean(rolloff)),
        "rolloff_std": float(np.std(rolloff)),
        "flatness_mean": float(np.mean(flatness)),
        "flatness_std": float(np.std(flatness)),
    }
    for idx in range(mfcc.shape[0]):
        features[f"mfcc_{idx + 1}_mean"] = float(np.mean(mfcc[idx]))
        features[f"mfcc_{idx + 1}_std"] = float(np.std(mfcc[idx]))
    return features

def build_feature_frame(df: pd.DataFrame, split: str, limit: int | None = None) -> pd.DataFrame:
    rows = []
    source = df.head(limit) if limit else df
    for _, row in source.iterrows():
        path = resolve_audio_path(row, split)
        if not path.exists():
            continue
        item = {"id": row["id"], "audio_path": str(path), **extract_audio_features(path)}
        for target in NUMERIC_TARGETS + ["clean_for_asr"]:
            item[target] = row[target]
        rows.append(item)
    return pd.DataFrame(rows)

# Start with a small limit for a smoke test, then set limit=None for the full run.
train_features = build_feature_frame(train_df, "train", limit=500)
val_features = build_feature_frame(val_df, "val", limit=200)
test_features = build_feature_frame(test_df, "test", limit=200)

print(train_features.shape, val_features.shape, test_features.shape)
display(train_features.head())

(500, 49) (200, 49) (200, 49)


,id,audio_path,duration,rms_mean,rms_std,zcr_mean,zcr_std,centroid_mean,centroid_std,bandwidth_mean,...,mfcc_13_mean,mfcc_13_std,Speed,Naturalness,Background noise,Distortion,Listening effort,Continuity,Overall quality,clean_for_asr
0,sys01tts-0000000.wav,data\QualiSpeech\wav_part1\wav\train\sys01tts-...,7.029375,0.036950,0.034542,0.150049,0.121470,1887.477132,1122.297325,1591.222265,...,-11.138203,9.133905,2,5,5,5,5,5,5,1
1,sys01tts-0000001.wav,data\QualiSpeech\wav_part1\wav\train\sys01tts-...,1.546687,0.038124,0.019955,0.157914,0.141716,2136.429436,1296.519156,1772.951711,...,-3.107710,7.838036,2,5,5,5,5,5,5,1
2,sys01tts-0000002.wav,data\QualiSpeech\wav_part1\wav\train\sys01tts-...,1.333375,0.059259,0.037436,0.313581,0.087239,3142.872072,959.162663,2021.435683,...,9.317739,8.755967,2,5,3,5,5,5,4,0
3,sys01tts-0000003.wav,data\QualiSpeech\wav_part1\wav\train\sys01tts-...,5.216000,0.052825,0.041792,0.154047,0.101962,2010.484604,930.548448,1771.442920,...,0.174621,7.493736,3,5,4,5,5,5,4,1
4,sys01tts-0000004.wav,data\QualiSpeech\wav_part1\wav\train\sys01tts-...,3.402687,0.089103,0.044639,0.215724,0.095264,2237.307236,628.472377,1516.512173,...,1.181801,6.741951,3,5,3,5,5,5,4,0


In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

if len(train_features) == 0:
    print("No local audio files were found. Construct/download the audio files first, then rerun the audio baseline cells.")
else:
    feature_cols = [c for c in train_features.columns if c not in ["id", "audio_path", "clean_for_asr"] + NUMERIC_TARGETS]
    clf = HistGradientBoostingClassifier(random_state=SEED)
    clf.fit(train_features[feature_cols], train_features["clean_for_asr"])
    pred = clf.predict(test_features[feature_cols])
    print(classification_report(test_features["clean_for_asr"], pred, target_names=["Re-record", "Clean for ASR"]))
    ConfusionMatrixDisplay(confusion_matrix(test_features["clean_for_asr"], pred), display_labels=["Re-record", "Clean"]).plot(cmap="Blues")
    plt.title("Audio Feature Baseline - Cleanliness Gate")
    plt.show()

               precision    recall  f1-score   support

    Re-record       0.28      0.58      0.38        48
Clean for ASR       0.80      0.53      0.63       152

     accuracy                           0.54       200
    macro avg       0.54      0.55      0.51       200
 weighted avg       0.68      0.54      0.57       200



C:\Users\ah625\AppData\Local\Temp\ipykernel_25344\238862368.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Deep Transfer-Learning Cleanliness Model

This is the main model direction for the voice-cleanliness phase. It follows the literature review more closely than the handcrafted-feature baseline: instead of manually summarizing the waveform, it uses a pretrained self-supervised speech encoder and fine-tunes task-specific heads.

The model has two outputs:

- **Regression head:** predicts the seven QualiSpeech numerical scores.
- **Classification head:** predicts whether the audio is clean enough for ASR.

Start with `QUICK_RUN = True` to verify everything works. Then set `QUICK_RUN = False` for the full experiment.

In [44]:
import math
import soundfile as sf
import torch
import torch.nn as nn
try:
    from safetensors.torch import load_file, save_file
    HAS_SAFETENSORS = True
except Exception:
    HAS_SAFETENSORS = False
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable, **kwargs):
        return iterable
from torch.utils.data import DataLoader, Dataset
from transformers import AutoFeatureExtractor, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, mean_absolute_error, classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support

DEEP_OUTPUT_DIR = OUTPUT_DIR / "deep_wav2vec_cleanliness_stratified"
DEEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENCODER_NAME = "facebook/wav2vec2-base-960h"
# This checkpoint has safetensors weights, avoiding torch<2.6 pickle-loading restrictions.
USE_SAFETENSORS = True
SAMPLE_RATE = 16000
MAX_AUDIO_SECONDS = 12
DEEP_BATCH_SIZE = 4
DEEP_EPOCHS = 10
DEEP_LR = 1e-5
DEEP_WARMUP_RATIO = 0.1
REGRESSION_LOSS_WEIGHT = 0.5
THRESHOLD_SELECTION_METRIC = "macro_f1"
FREEZE_ENCODER = False
QUICK_RUN = True
QUICK_RANDOM_SEED = 42
QUICK_TRAIN_LIMIT = 800
QUICK_VAL_LIMIT = 200
QUICK_TEST_LIMIT = 200

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


In [45]:
def stratified_quick_sample(df: pd.DataFrame, limit: int | None, label_col: str = "clean_for_asr") -> pd.DataFrame:
    if limit is None or limit >= len(df):
        return df.sample(frac=1.0, random_state=QUICK_RANDOM_SEED).reset_index(drop=True)
    sampled_parts = []
    for label, group in df.groupby(label_col):
        n = max(1, round(limit * len(group) / len(df)))
        sampled_parts.append(group.sample(n=min(n, len(group)), random_state=QUICK_RANDOM_SEED))
    sampled = pd.concat(sampled_parts).sample(frac=1.0, random_state=QUICK_RANDOM_SEED).reset_index(drop=True)
    if len(sampled) > limit:
        sampled = sampled.sample(n=limit, random_state=QUICK_RANDOM_SEED).reset_index(drop=True)
    return sampled


def prepare_audio_dataframe(df: pd.DataFrame, split: str, limit: int | None = None) -> pd.DataFrame:
    rows = []
    source = stratified_quick_sample(df, limit) if QUICK_RUN else df
    for _, row in source.iterrows():
        path = resolve_audio_path(row, split)
        if path.exists():
            item = row.copy()
            item["audio_path"] = str(path)
            rows.append(item)
    result = pd.DataFrame(rows).reset_index(drop=True)
    print(f"{split}: {len(result):,} usable audio rows")
    print(result["clean_for_asr"].value_counts().sort_index().rename({0: "Re-record", 1: "Clean for ASR"}))
    return result

if QUICK_RUN:
    deep_train_df = prepare_audio_dataframe(train_df, "train", QUICK_TRAIN_LIMIT)
    deep_val_df = prepare_audio_dataframe(val_df, "val", QUICK_VAL_LIMIT)
    deep_test_df = prepare_audio_dataframe(test_df, "test", QUICK_TEST_LIMIT)
else:
    deep_train_df = prepare_audio_dataframe(train_df, "train", None)
    deep_val_df = prepare_audio_dataframe(val_df, "val", None)
    deep_test_df = prepare_audio_dataframe(test_df, "test", None)

assert len(deep_train_df) > 0, "No training audio files found. Check AUDIO_ROOT and resolve_audio_path()."
assert len(deep_val_df) > 0, "No validation audio files found. Check AUDIO_ROOT and resolve_audio_path()."
assert len(deep_test_df) > 0, "No test audio files found. Check AUDIO_ROOT and resolve_audio_path()."


train: 539 usable audio rows
clean_for_asr
Re-record        370
Clean for ASR    169
Name: count, dtype: int64
val: 131 usable audio rows
clean_for_asr
Re-record        97
Clean for ASR    34
Name: count, dtype: int64
test: 121 usable audio rows
clean_for_asr
Re-record        66
Clean for ASR    55
Name: count, dtype: int64


In [46]:
def load_audio(path: str, target_sr: int = SAMPLE_RATE, max_seconds: int = MAX_AUDIO_SECONDS) -> np.ndarray:
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)

    if sr != target_sr:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)

    max_len = target_sr * max_seconds
    if len(audio) > max_len:
        audio = audio[:max_len]

    peak = np.max(np.abs(audio)) if len(audio) else 0.0
    if peak > 0:
        audio = audio / peak
    return audio


class QualiSpeechAudioDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio = load_audio(row["audio_path"])
        # Normalize 1-5 quality scores to 0-1 for stable regression.
        score_targets = ((row[NUMERIC_TARGETS].astype(float).values - 1.0) / 4.0).astype(np.float32)
        clean_target = np.float32(row["clean_for_asr"])
        return {
            "audio": audio,
            "score_targets": score_targets,
            "clean_target": clean_target,
            "id": row["id"],
        }

feature_extractor = AutoFeatureExtractor.from_pretrained(ENCODER_NAME)


def collate_audio_batch(batch):
    audios = [item["audio"] for item in batch]
    encoded = feature_extractor(
        audios,
        sampling_rate=SAMPLE_RATE,
        padding=True,
        return_attention_mask=True,
        return_tensors="pt",
    )
    return {
        "input_values": encoded["input_values"],
        "attention_mask": encoded.get("attention_mask"),
        "score_targets": torch.tensor(np.stack([item["score_targets"] for item in batch]), dtype=torch.float32),
        "clean_target": torch.tensor([item["clean_target"] for item in batch], dtype=torch.float32),
        "ids": [item["id"] for item in batch],
    }

train_audio_ds = QualiSpeechAudioDataset(deep_train_df)
val_audio_ds = QualiSpeechAudioDataset(deep_val_df)
test_audio_ds = QualiSpeechAudioDataset(deep_test_df)

train_audio_loader = DataLoader(train_audio_ds, batch_size=DEEP_BATCH_SIZE, shuffle=True, collate_fn=collate_audio_batch)
val_audio_loader = DataLoader(val_audio_ds, batch_size=DEEP_BATCH_SIZE, shuffle=False, collate_fn=collate_audio_batch)
test_audio_loader = DataLoader(test_audio_ds, batch_size=DEEP_BATCH_SIZE, shuffle=False, collate_fn=collate_audio_batch)


In [47]:
class SpeechCleanlinessModel(nn.Module):
    def __init__(self, encoder_name: str, num_scores: int = len(NUMERIC_TARGETS), freeze_encoder: bool = False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            encoder_name,
            use_safetensors=USE_SAFETENSORS,
        )
        hidden_size = self.encoder.config.hidden_size
        self.regression_head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_scores),
        )
        self.clean_head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 1),
        )
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def forward(self, input_values, attention_mask=None):
        outputs = self.encoder(input_values=input_values, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        if attention_mask is not None:
            # Convert sample-level mask to frame-level length approximately by pooling over valid hidden states.
            pooled = hidden.mean(dim=1)
        else:
            pooled = hidden.mean(dim=1)
        score_pred = self.regression_head(pooled)
        clean_logit = self.clean_head(pooled).squeeze(-1)
        return score_pred, clean_logit

model = SpeechCleanlinessModel(ENCODER_NAME, freeze_encoder=FREEZE_ENCODER).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=DEEP_LR)
total_steps = len(train_audio_loader) * DEEP_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * DEEP_WARMUP_RATIO),
    num_training_steps=total_steps,
)

regression_loss_fn = nn.MSELoss()

# The dataset is usually imbalanced, so a plain BCE loss can learn the lazy strategy:
# predict most clips as clean. These weights make mistakes on each class count more equally.
train_class_counts = deep_train_df["clean_for_asr"].value_counts().to_dict()
num_train_examples = len(deep_train_df)
weight_rerecord = num_train_examples / (2.0 * max(train_class_counts.get(0, 0), 1))
weight_clean = num_train_examples / (2.0 * max(train_class_counts.get(1, 0), 1))
CLEAN_CLASS_WEIGHTS = torch.tensor([weight_rerecord, weight_clean], dtype=torch.float32, device=DEVICE)
classification_loss_fn = nn.BCEWithLogitsLoss(reduction="none")
print("Class counts:", train_class_counts)
print("Class weights [Re-record, Clean]:", CLEAN_CLASS_WEIGHTS.detach().cpu().numpy().round(3).tolist())


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class counts: {0: 370, 1: 169}
Class weights [Re-record, Clean]: [0.7279999852180481, 1.5950000286102295]


In [48]:
if "THRESHOLD_SELECTION_METRIC" not in globals():
    THRESHOLD_SELECTION_METRIC = "macro_f1"

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    mean_absolute_error,
    precision_recall_fscore_support,
)


def train_cleanliness_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_items = 0

    for batch in tqdm(loader, desc="train", leave=False):
        optimizer.zero_grad(set_to_none=True)
        input_values = batch["input_values"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE) if batch["attention_mask"] is not None else None
        score_targets = batch["score_targets"].to(DEVICE)
        clean_target = batch["clean_target"].to(DEVICE)

        score_pred, clean_logit = model(input_values=input_values, attention_mask=attention_mask)
        reg_loss = regression_loss_fn(score_pred, score_targets)
        cls_loss_per_item = classification_loss_fn(clean_logit, clean_target)
        sample_weights = torch.where(clean_target >= 0.5, CLEAN_CLASS_WEIGHTS[1], CLEAN_CLASS_WEIGHTS[0])
        cls_loss = (cls_loss_per_item * sample_weights).mean()
        loss = cls_loss + REGRESSION_LOSS_WEIGHT * reg_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * input_values.size(0)
        total_items += input_values.size(0)

    return total_loss / total_items


def calculate_cleanliness_metrics(y_true, y_prob, threshold: float = 0.5):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    precision_by_class, recall_by_class, f1_by_class, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )
    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "rerecord_precision": precision_by_class[0],
        "rerecord_recall": recall_by_class[0],
        "rerecord_f1": f1_by_class[0],
        "clean_precision": precision_by_class[1],
        "clean_recall": recall_by_class[1],
        "clean_f1": f1_by_class[1],
        "roc_auc": roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else np.nan,
    }
    return metrics, y_pred


def find_best_cleanliness_threshold(y_true, y_prob, metric: str | None = None):
    if metric is None:
        metric = THRESHOLD_SELECTION_METRIC
    rows = []
    for threshold in np.linspace(0.05, 0.95, 91):
        metrics, _ = calculate_cleanliness_metrics(y_true, y_prob, threshold=float(threshold))
        rows.append(metrics)
    threshold_df = pd.DataFrame(rows)
    # Prefer the selected metric; tie-break using re-record recall because missing bad audio is costly.
    threshold_df = threshold_df.sort_values([metric, "rerecord_recall", "balanced_accuracy"], ascending=False)
    best = threshold_df.iloc[0].to_dict()
    return best, threshold_df


@torch.no_grad()
def evaluate_cleanliness_model(model, loader, threshold: float = 0.5):
    model.eval()
    all_clean_true, all_clean_prob, all_clean_pred = [], [], []
    all_score_true, all_score_pred = [], []
    all_ids = []

    for batch in tqdm(loader, desc="eval", leave=False):
        input_values = batch["input_values"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE) if batch["attention_mask"] is not None else None
        score_targets = batch["score_targets"].numpy()
        clean_target = batch["clean_target"].numpy()

        score_pred, clean_logit = model(input_values=input_values, attention_mask=attention_mask)
        clean_prob = torch.sigmoid(clean_logit).detach().cpu().numpy()
        score_pred_np = score_pred.detach().cpu().numpy()

        all_clean_true.extend(clean_target.astype(int))
        all_clean_prob.extend(clean_prob)
        all_score_true.append(score_targets)
        all_score_pred.append(score_pred_np)
        all_ids.extend(batch["ids"])

    score_true = np.vstack(all_score_true)
    score_pred = np.vstack(all_score_pred)
    # Convert normalized 0-1 predictions back to the original 1-5 score scale.
    score_true_1_5 = score_true * 4.0 + 1.0
    score_pred_1_5 = np.clip(score_pred, 0, 1) * 4.0 + 1.0

    metrics, all_clean_pred = calculate_cleanliness_metrics(all_clean_true, all_clean_prob, threshold=threshold)
    metrics["score_mae"] = mean_absolute_error(score_true_1_5, score_pred_1_5)
    predictions = pd.DataFrame({
        "id": all_ids,
        "clean_true": all_clean_true,
        "clean_probability": all_clean_prob,
        "clean_prediction": all_clean_pred,
    })
    for idx, target in enumerate(NUMERIC_TARGETS):
        predictions[f"true_{target}"] = score_true_1_5[:, idx]
        predictions[f"pred_{target}"] = score_pred_1_5[:, idx]
    return metrics, predictions


In [49]:
def save_model_state(model, path):
    if HAS_SAFETENSORS and path.suffix == ".safetensors":
        save_file(model.state_dict(), str(path))
    else:
        torch.save(model.state_dict(), path)


def load_model_state(path, device):
    if path.suffix == ".safetensors":
        return load_file(str(path), device=str(device))
    return torch.load(path, map_location=device, weights_only=True)


best_val_macro_f1 = -1.0
best_threshold = 0.5
history = []
best_path = DEEP_OUTPUT_DIR / ("best_speech_cleanliness_model.safetensors" if HAS_SAFETENSORS else "best_speech_cleanliness_model.pt")

for epoch in range(1, DEEP_EPOCHS + 1):
    train_loss = train_cleanliness_epoch(model, train_audio_loader)
    val_metrics_at_05, val_predictions_at_05 = evaluate_cleanliness_model(model, val_audio_loader, threshold=0.5)
    best_threshold_metrics, threshold_grid = find_best_cleanliness_threshold(
        val_predictions_at_05["clean_true"].to_numpy(),
        val_predictions_at_05["clean_probability"].to_numpy(),
    )
    val_metrics, val_predictions = evaluate_cleanliness_model(model, val_audio_loader, threshold=best_threshold_metrics["threshold"])
    row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
    row["val_f1_at_0_5"] = val_metrics_at_05["f1"]
    row["val_macro_f1_at_0_5"] = val_metrics_at_05["macro_f1"]
    row["threshold_selection_metric"] = THRESHOLD_SELECTION_METRIC
    history.append(row)
    pd.DataFrame(history).to_csv(DEEP_OUTPUT_DIR / "training_history.csv", index=False)
    threshold_grid.to_csv(DEEP_OUTPUT_DIR / f"threshold_grid_epoch_{epoch}.csv", index=False)
    print(row)

    if val_metrics[THRESHOLD_SELECTION_METRIC] > best_val_macro_f1:
        best_val_macro_f1 = val_metrics[THRESHOLD_SELECTION_METRIC]
        best_threshold = float(best_threshold_metrics["threshold"])
        save_model_state(model, best_path)
        feature_extractor.save_pretrained(DEEP_OUTPUT_DIR / "feature_extractor")
        with open(DEEP_OUTPUT_DIR / "best_threshold.json", "w", encoding="utf-8") as f:
            json.dump({"threshold": best_threshold, "metric": THRESHOLD_SELECTION_METRIC, "validation_metrics": val_metrics}, f, indent=2)
        print(f"Saved best model at epoch {epoch} with val {THRESHOLD_SELECTION_METRIC}={best_val_macro_f1:.4f} and threshold={best_threshold:.2f}")

pd.DataFrame(history)


train:   0%|          | 0/135 [00:00<?, ?it/s]

eval:   0%|          | 0/33 [00:00<?, ?it/s]

eval:   0%|          | 0/33 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.9265942341321475, 'val_threshold': 0.5199999999999999, 'val_accuracy': 0.5419847328244275, 'val_balanced_accuracy': 0.6716191631291692, 'val_precision': 0.35555555555555557, 'val_recall': 0.9411764705882353, 'val_f1': 0.5161290322580645, 'val_macro_f1': 0.5406732117812061, 'val_rerecord_precision': 0.9512195121951219, 'val_rerecord_recall': 0.4020618556701031, 'val_rerecord_f1': 0.5652173913043478, 'val_clean_precision': 0.35555555555555557, 'val_clean_recall': 0.9411764705882353, 'val_clean_f1': 0.5161290322580645, 'val_roc_auc': 0.7798665858095816, 'val_score_mae': 1.6007604598999023, 'val_f1_at_0_5': 0.422360248447205, 'val_macro_f1_at_0_5': 0.2507840846196421, 'threshold_selection_metric': 'macro_f1'}
Saved best model at epoch 1 with val macro_f1=0.5407 and threshold=0.52


train:   0%|          | 0/135 [00:00<?, ?it/s]

...

train:   0%|          | 0/135 [00:00<?, ?it/s]

eval:   0%|          | 0/33 [00:00<?, ?it/s]

eval:   0%|          | 0/33 [00:00<?, ?it/s]

{'epoch': 10, 'train_loss': 0.5399834385387088, 'val_threshold': 0.83, 'val_accuracy': 0.6946564885496184, 'val_balanced_accuracy': 0.6791995148574894, 'val_precision': 0.44, 'val_recall': 0.6470588235294118, 'val_f1': 0.5238095238095238, 'val_macro_f1': 0.6495452113429642, 'val_rerecord_precision': 0.8518518518518519, 'val_rerecord_recall': 0.711340206185567, 'val_rerecord_f1': 0.7752808988764045, 'val_clean_precision': 0.44, 'val_clean_recall': 0.6470588235294118, 'val_clean_f1': 0.5238095238095238, 'val_roc_auc': 0.7768344451182535, 'val_score_mae': 0.7870572805404663, 'val_f1_at_0_5': 0.56, 'val_macro_f1_at_0_5': 0.6441975308641976, 'threshold_selection_metric': 'macro_f1'}


,epoch,train_loss,val_threshold,val_accuracy,val_balanced_accuracy,val_precision,val_recall,val_f1,val_macro_f1,val_rerecord_precision,val_rerecord_recall,val_rerecord_f1,val_clean_precision,val_clean_recall,val_clean_f1,val_roc_auc,val_score_mae,val_f1_at_0_5,val_macro_f1_at_0_5,threshold_selection_metric
0,1,0.926594,0.52,0.541985,0.671619,0.355556,0.941176,0.516129,0.540673,0.951220,0.402062,0.565217,0.355556,0.941176,0.516129,0.779867,1.600760,0.422360,0.250784,macro_f1
1,2,0.759709,0.51,0.671756,0.654184,0.411765,0.617647,0.494118,0.625590,0.837500,0.690722,0.757062,0.411765,0.617647,0.494118,0.750758,0.842063,0.553571,0.610119,macro_f1
2,3,0.707230,0.56,0.725191,0.728472,0.480769,0.735294,0.581395,0.688425,0.886076,0.721649,0.795455,0.480769,0.735294,0.581395,0.793208,0.867371,0.555556,0.621934,macro_f1
3,4,0.662268,0.59,0.671756,0.692389,0.423729,0.735294,0.537634,0.641598,0.875000,0.649485,0.745562,0.423729,0.735294,0.537634,0.757126,0.820687,0.519231,0.601388,macro_f1
4,5,0.627775,0.69,0.793893,0.698454,0.629630,0.500000,0.557377,0.711524,0.836538,0.896907,0.865672,0.629630,0.500000,0.557377,0.732868,0.817352,0.550000,0.676099,macro_f1
5,6,0.629908,0.58,0.809160,0.737417,0.645161,0.588235,0.615385,0.744241,0.860000,0.886598,0.873096,0.645161,0.588235,0.615385,0.744391,0.803664,0.608696,0.734400,macro_f1
6,7,0.573461,0.79,0.694656,0.707853,0.446429,0.735294,0.555556,0.661499,0.880000,0.680412,0.767442,0.446429,0.735294,0.555556,0.796240,0.810187,0.563636,0.623923,macro_f1
7,8,0.573909,0.82,0.717557,0.704215,0.469388,0.676471,0.554217,0.673756,0.865854,0.731959,0.793296,0.469388,0.676471,0.554217,0.787750,0.797794,0.566038,0.635583,macro_f1
8,9,0.576618,0.83,0.709924,0.689509,0.458333,0.647059,0.536585,0.662737,0.855422,0.731959,0.788889,0.458333,0.647059,0.536585,0.770770,0.787251,0.554455,0.637476,macro_f1
9,10,0.539983,0.83,0.694656,0.679200,0.440000,0.647059,0.523810,0.649545,0.851852,0.711340,0.775281,0.440000,0.647059,0.523810,0.776834,0.787057,0.560000,0.644198,macro_f1


In [50]:
model.load_state_dict(load_model_state(best_path, DEVICE))
threshold_path = DEEP_OUTPUT_DIR / "best_threshold.json"
if threshold_path.exists():
    with open(threshold_path, "r", encoding="utf-8") as f:
        best_threshold_info = json.load(f)
    test_threshold = float(best_threshold_info["threshold"])
else:
    test_threshold = 0.5

test_metrics, test_predictions = evaluate_cleanliness_model(model, test_audio_loader, threshold=test_threshold)

pd.DataFrame([test_metrics]).to_csv(DEEP_OUTPUT_DIR / "test_metrics.csv", index=False)
test_predictions.to_csv(DEEP_OUTPUT_DIR / "test_predictions.csv", index=False)

print("Decision threshold:", test_threshold)
print(test_metrics)
print(classification_report(test_predictions["clean_true"], test_predictions["clean_prediction"], target_names=["Re-record", "Clean for ASR"]))

cm = confusion_matrix(test_predictions["clean_true"], test_predictions["clean_prediction"])
ConfusionMatrixDisplay(cm, display_labels=["Re-record", "Clean"]).plot(cmap="Blues")
plt.title("Deep Speech Cleanliness Model - Test Confusion Matrix")
plt.show()


eval:   0%|          | 0/31 [00:00<?, ?it/s]

Decision threshold: 0.58
{'threshold': 0.58, 'accuracy': 0.7355371900826446, 'balanced_accuracy': 0.7151515151515152, 'precision': 0.8709677419354839, 'recall': 0.4909090909090909, 'f1': 0.627906976744186, 'macro_f1': 0.7113893858079905, 'rerecord_precision': 0.6888888888888889, 'rerecord_recall': 0.9393939393939394, 'rerecord_f1': 0.7948717948717948, 'clean_precision': 0.8709677419354839, 'clean_recall': 0.4909090909090909, 'clean_f1': 0.627906976744186, 'roc_auc': 0.7570247933884299, 'score_mae': 0.9730421900749207}
               precision    recall  f1-score   support

    Re-record       0.69      0.94      0.79        66
Clean for ASR       0.87      0.49      0.63        55

     accuracy                           0.74       121
    macro avg       0.78      0.72      0.71       121
 weighted avg       0.77      0.74      0.72       121



C:\Users\ah625\AppData\Local\Temp\ipykernel_25344\3213297532.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [51]:
# Visualize threshold selection and per-class performance for the thesis/report.
threshold_files = sorted(DEEP_OUTPUT_DIR.glob("threshold_grid_epoch_*.csv"))
if threshold_files:
    threshold_df = pd.read_csv(threshold_files[-1])
    plt.figure(figsize=(9, 5))
    plt.plot(threshold_df["threshold"], threshold_df["macro_f1"], label="Macro F1")
    plt.plot(threshold_df["threshold"], threshold_df["rerecord_recall"], label="Re-record recall")
    plt.plot(threshold_df["threshold"], threshold_df["balanced_accuracy"], label="Balanced accuracy")
    plt.axvline(test_threshold, color="black", linestyle="--", label=f"Selected threshold = {test_threshold:.2f}")
    plt.xlabel("Clean-class probability threshold")
    plt.ylabel("Score")
    plt.title("Validation Threshold Selection")
    plt.legend()
    plt.tight_layout()
    plt.savefig(DEEP_OUTPUT_DIR / "threshold_selection_curve.png", dpi=200)
    plt.show()

class_f1 = pd.DataFrame({
    "class": ["Re-record", "Clean for ASR"],
    "f1": [test_metrics["rerecord_f1"], test_metrics["clean_f1"]],
    "recall": [test_metrics["rerecord_recall"], test_metrics["clean_recall"]],
})
ax = class_f1.plot(x="class", y=["f1", "recall"], kind="bar", figsize=(7, 4), ylim=(0, 1), rot=0)
ax.set_title("Test Performance by Class")
ax.set_ylabel("Score")
plt.tight_layout()
plt.savefig(DEEP_OUTPUT_DIR / "class_performance.png", dpi=200)
plt.show()


C:\Users\ah625\AppData\Local\Temp\ipykernel_25344\3471935499.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\ah625\AppData\Local\Temp\ipykernel_25344\3471935499.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [52]:
with open(DEEP_OUTPUT_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "encoder_name": ENCODER_NAME,
            "sample_rate": SAMPLE_RATE,
            "max_audio_seconds": MAX_AUDIO_SECONDS,
            "batch_size": DEEP_BATCH_SIZE,
            "epochs": DEEP_EPOCHS,
            "learning_rate": DEEP_LR,
            "warmup_ratio": DEEP_WARMUP_RATIO,
            "regression_loss_weight": REGRESSION_LOSS_WEIGHT,
            "threshold_selection_metric": THRESHOLD_SELECTION_METRIC,
            "freeze_encoder": FREEZE_ENCODER,
            "quick_run": QUICK_RUN,
            "quick_train_limit": QUICK_TRAIN_LIMIT if QUICK_RUN else None,
            "quick_val_limit": QUICK_VAL_LIMIT if QUICK_RUN else None,
            "quick_test_limit": QUICK_TEST_LIMIT if QUICK_RUN else None,
            "numeric_targets": NUMERIC_TARGETS,
            "class_weighting": "inverse_frequency_per_batch_item",
        },
        f,
        indent=2,
    )


## Transfer-Learning Model Direction

For the thesis-level model, the most defensible next step is a pretrained speech encoder with regression/classification heads. This follows the same motivation as UTMOS and recent non-intrusive speech assessment systems: use a pretrained acoustic representation and fine-tune it for perceptual quality.

Recommended architecture:

- Encoder: `facebook/wav2vec2-base` or `microsoft/wavlm-base-plus`
- Head 1: seven-output regression head for the QualiSpeech numerical scores
- Head 2: binary classification head for `clean_for_asr`
- Loss: MSE for score regression + class-weighted BCE for cleanliness classification
- Selection metric: validation macro F1 with threshold tuning, plus explicit recall/F1 for the `Re-record` class and MAE for numerical scores

This is more aligned with the literature than training a speech-to-text model from scratch. The ASR component can later use pretrained Whisper; this module only decides whether the audio quality is acceptable for ASR.

## Planned Thesis Wording

The voice cleanliness module is treated as a non-intrusive speech quality assessment task rather than an ASR task. This is consistent with the literature reviewed in the thesis: MOSNet demonstrates the use of deep acoustic features for naturalness prediction, NISQA extends this to multidimensional quality scores using CNN/self-attention, and UTMOS shows the importance of transfer learning for robust MOS prediction. QualiSpeech is particularly suitable for this project because it provides both numerical low-level quality labels and natural-language quality descriptions, allowing the system to reason about noise, distortion, continuity, listening effort, and overall quality. The output of this module acts as a gate before transcription, reducing the risk that noisy or distorted recordings produce poor transcripts and therefore false coherence errors.